# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #4 — "The Freshness Multiplier."** The paper's headline number: 365+ day content that
was refreshed within 30 days shows a **3.2x health boost** (10.7 → 34.5) and **57x more
impressions** (71 → 4,039), framed as "one of the strongest measured levers available."

*My methodology question, respectfully:* where does "refreshed" come from as a group label? The
paper compares refreshed vs not-refreshed pages within the 365+ age bucket, but it never says
whether refresh assignment was random or chosen by an editor. If editors pick *which* old pages
are worth refreshing, they are likely already picking pages with some remaining proven demand or
strategic value — the same selection-bias pattern `writing-honest-claims` names directly
("if the treated group was CHOSEN, part of the gap is the choosing, not the treatment"). This is a
cross-sectional, single-snapshot comparison, not an experiment with random assignment, so the
honest form of this finding is "refreshed 365+ pages in this portfolio look much healthier than
un-refreshed ones" (observed association) rather than "refreshing causes a 3.2x boost" (the paper
mostly avoids the causal word, and I want to flag that as a strength worth keeping, not weakening).

**ML Appendix — "What Predicts Growth?"** Logistic regression, reported at **71% holdout
accuracy**, with content age as the strongest negative signal and "recent impressions" among the
strongest positive signals for separating growing vs declining pages.

*My methodology question, respectfully:* two things I can't verify from the public paper alone.
First — where does the growth/decline label come from? The paper states elsewhere that
"Trend Direction" is "calculated from 30d-vs-prev-30d impression change." If "recent impressions"
in the feature list means the same last-30-day window the label is built from (rather than the
full 90-day window), that is the exact label-derived-feature trap `hunting-leakage-and-validating`
warns about — and it's the same trap I had to exclude in my own Week-5 feature set
(`impressions_last_30d` / `impressions_prev_30d` are the label's own numerator/denominator here).
Second — the methodology page doesn't say whether the 80/20 holdout split is random-by-row or
grouped by brand. With 57 brands likely sharing house style/CMS per brand, a plain random split
could let same-brand pages leak across train/test and inflate 71% above what a genuinely new brand
would see (Section 2 below shows exactly this kind of gap on my own data). Neither question is a
"gotcha" — the paper is transparent that ML pages are exploratory appendix material, not headline
evidence — but both are the two questions I'd want asked of my own 71%-style number before trusting it.

In [1]:
# Sanity-check the arithmetic behind Finding #4's headline ratios (does the math the paper
# reports actually reduce to 3.2x and 57x, or is there rounding drama hiding in the words?)
health_before, health_after = 10.7, 34.5
imp_before, imp_after = 71, 4039

health_ratio = health_after / health_before
imp_ratio = imp_after / imp_before

print(f"Health ratio: {health_after} / {health_before} = {health_ratio:.2f}x  (paper says 3.2x)")
print(f"Impression ratio: {imp_after} / {imp_before} = {imp_ratio:.1f}x  (paper says 57x)")
print()
print("Both check out arithmetically -- the numbers behind the headline ratios are internally consistent.")
print("What the arithmetic can't tell me: whether 'refreshed' pages were a random sample of the 365+")
print("bucket or a hand-picked one. That's the methodology question above, not a math error.")

Health ratio: 34.5 / 10.7 = 3.22x  (paper says 3.2x)
Impression ratio: 4039 / 71 = 56.9x  (paper says 57x)

Both check out arithmetically -- the numbers behind the headline ratios are internally consistent.
What the arithmetic can't tell me: whether 'refreshed' pages were a random sample of the 365+
bucket or a hand-picked one. That's the methodology question above, not a math error.


## 2. My model under an honest split (before/after)

My Week-5 model (`w05_model.ipynb`) already used a client-grouped split, reasoning that pages from
the same client can share a CMS template or writer, so a random row split would let the model
memorize a client's house style rather than learn a real refresh signal. Here I make that
reasoning verifiable instead of asserted: I train the **same Logistic Regression, same feature
set, same data**, once on a naive random 80/20 row split (the "before" — what I'd get if I hadn't
thought about grouping) and once on the client-grouped 80/20 split (the "after" — what `w05`
actually reports). Same `random_state`, same K values, same metric family, so the only thing that
changes between rows is the split design.

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

# Same leakage-safe feature set as w05_model.ipynb -- see that notebook for the full exclusion list
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]

num_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
enc_frame = pd.get_dummies(cat_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num_frame.reset_index(drop=True), enc_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
groups = df["client_id"]

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    k = min(k, len(y_true))
    return y_true[order[:k]].mean()

def run_split(train_idx, test_idx, label):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    logreg = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ])
    logreg.fit(X_train, y_train)
    proba = logreg.predict_proba(X_test)[:, 1]
    row = {"split": label, "n_test_rows": len(test_idx), "test_base_rate": round(y_test.mean(), 3)}
    for k in [20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(y_test, proba, k), 3)
    row["roc_auc"] = round(roc_auc_score(y_test, proba), 3)
    row["avg_precision"] = round(average_precision_score(y_test, proba), 3)
    return row

# BEFORE: naive random row split -- ignores that rows repeat by client
rand_splitter = ShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
rand_train_idx, rand_test_idx = next(rand_splitter.split(X, y))
row_before = run_split(rand_train_idx, rand_test_idx, "BEFORE -- random row split")

# AFTER: client-grouped split -- the honest design, matches w05_model.ipynb
grp_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
grp_train_idx, grp_test_idx = next(grp_splitter.split(X, y, groups))
row_after = run_split(grp_train_idx, grp_test_idx, "AFTER -- client-grouped split")

rand_train_clients = set(groups.iloc[rand_train_idx])
rand_test_clients = set(groups.iloc[rand_test_idx])
grp_train_clients = set(groups.iloc[grp_train_idx])
grp_test_clients = set(groups.iloc[grp_test_idx])

print(f"Total clients in dataset: {df['client_id'].nunique()}")
print(f"Random split  -- clients on BOTH sides: {len(rand_train_clients & rand_test_clients)}")
print(f"Grouped split -- clients on BOTH sides: {len(grp_train_clients & grp_test_clients)}  (should be 0)")
print()
comparison = pd.DataFrame([row_before, row_after])
comparison

Total clients in dataset: 32
Random split  -- clients on BOTH sides: 31
Grouped split -- clients on BOTH sides: 0  (should be 0)



,split,n_test_rows,test_base_rate,precision@20,precision@50,precision@100,roc_auc,avg_precision
0,BEFORE -- random row split,6000,0.545,1.00,0.92,0.89,0.703,0.719
1,AFTER -- client-grouped split,6163,0.511,0.65,0.72,0.66,0.583,0.577


**Reading the before/after table.** The random-row split lets 31 of 32 clients appear on both
sides -- the model can partly recognize a client's house style rather than learn a general refresh
signal. That inflates every metric: precision@20 hits a suspicious 1.00 (perfect), and ROC-AUC
reads 0.703. Once the split is grouped by client so **zero** clients repeat across train and test,
precision@20 drops to roughly 0.65 and ROC-AUC drops to roughly 0.58 -- a meaningful decline, not
a rounding difference.

That gap **is** the finding, per `hunting-leakage-and-validating`: it's not that the model is
"worse" under the honest split, it's that the random-split number was partly measuring
client-memorization rather than skill on genuinely unseen pages. The honest number (~0.58 ROC-AUC,
modest precision@K lift over the ~51% base rate) is lower, smaller, and true -- which is exactly
why `w05_model.ipynb` used the grouped split from the start rather than reporting the inflated one.

## 3. Leakage audit

*The same hunt from Week 3, on my final feature set.* My Week-5 feature set already excludes
`trend_direction` / `trend_pct` (the label source) and the six last-30d/prev-30d columns
(`trend_pct`'s own numerator and denominator) as label-derived. Rather than just asserting that
exclusion was correct, I run the checklist's own verification method: deliberately add the
suspected leaky columns back and confirm the score visibly jumps -- if it doesn't move, my test
harness itself would be broken and I couldn't trust any of the "clean" numbers above either.

In [3]:
from sklearn.model_selection import GroupShuffleSplit as _GSS  # re-import for clarity in this cell

# Re-use the client-grouped split and clean feature matrix X from Section 2 (the honest design)
def score_feature_matrix(X_variant, label):
    X_train, X_test = X_variant.iloc[grp_train_idx], X_variant.iloc[grp_test_idx]
    y_train, y_test = y.iloc[grp_train_idx], y.iloc[grp_test_idx]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    print(f"{label}: ROC-AUC = {auc:.3f}")
    return auc

print("Attack-the-model test -- add the suspected leaky columns back and watch for a jump:\n")
auc_clean = score_feature_matrix(X, "WITHOUT suspects (final Week-5 feature set, this notebook's Section 2 result)")

leaky_cols = [
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]
leaky_frame = df[leaky_cols].apply(pd.to_numeric, errors="coerce").fillna(0).reset_index(drop=True)
X_with_leak = pd.concat([X, leaky_frame], axis=1)
auc_leaky = score_feature_matrix(X_with_leak, "WITH suspects added back (trend_pct's own numerator/denominator)")

print(f"\nCollapse test: {auc_leaky:.3f} (with leak) drops to {auc_clean:.3f} (without) "
      f"-- a {auc_leaky - auc_clean:.3f} point fall.")
print("The jump confirms the test harness correctly detects leakage, and confirms excluding")
print("these six columns from the final feature set was the right call, not over-caution.")

Attack-the-model test -- add the suspected leaky columns back and watch for a jump:



WITHOUT suspects (final Week-5 feature set, this notebook's Section 2 result): ROC-AUC = 0.583


WITH suspects added back (trend_pct's own numerator/denominator): ROC-AUC = 0.848

Collapse test: 0.848 (with leak) drops to 0.583 (without) -- a 0.265 point fall.
The jump confirms the test harness correctly detects leakage, and confirms excluding
these six columns from the final feature set was the right call, not over-caution.


## 4. Claim rewrite

**My boldest sentence, from `w05_model.ipynb`'s feature-importance discussion:**

> "`content_age_days` showing up high is a fair, expected pattern too: older content has simply
> had more time to decay."

This states a causal mechanism ("has simply had more time to decay") as settled fact, from a
single cross-sectional snapshot with no repeated observations of the same page over time. Running
the numbers behind that sentence turns up something worth catching: the plain-English direction it
implies is actually backwards in this data.

In [4]:
print("Rows:", len(df), " Unique content_id:", df['content_id'].nunique())
print("One row per content item, no repeated observations over time:", len(df) == df['content_id'].nunique())
print()

age_by_label = df.groupby('is_declining_label')['content_age_days'].mean().round(1)
corr = df['content_age_days'].corr(df['is_declining_label'])

print("Mean content_age_days by label:")
print(age_by_label.rename({0: 'not declining', 1: 'declining'}))
print(f"\nCorrelation(content_age_days, is_declining_label): {corr:.3f}")

Rows: 30000  Unique content_id: 30000
One row per content item, no repeated observations over time: True

Mean content_age_days by label:
is_declining_label
not declining    279.8
declining        236.2
Name: content_age_days, dtype: float64

Correlation(content_age_days, is_declining_label): -0.164


**What the numbers actually show:** declining pages average 236 days old vs 280 for
non-declining pages -- correlation -0.16. In this dataset, older content is *slightly less* likely
to be labeled declining, not more. `content_age_days` is still a real, legitimate top-3 feature
importance in the Random Forest (importance is about how much a feature helps split the data, not
which direction it pushes on its own -- especially with non-linear interactions), but my plain-English
sentence asserted a specific direction and mechanism that this simple check doesn't support.

**Rewritten, safe version:**

> "`content_age_days` ranks among the top three feature importances in both the impurity-based
> and permutation-importance methods -- a real, replicated signal in this model, not an artifact.
> But this dataset is a single cross-sectional snapshot with one row per page and no repeated
> observations of the same page over time, so I can't distinguish 'aging causes decline' from
> 'the pages that happen to be declining in this snapshot happen to be somewhat younger.' A simple
> mean-by-label check even points the opposite direction of the causal story I originally told
> (declining pages average 236 days vs 280 for non-declining, r=-0.16), which is itself useful:
> it's a reminder that a feature can matter to a model's decisions without that feature's
> univariate direction matching a tidy narrative. The honest claim is *observed* importance, not
> a decay mechanism."

This is exactly the check `writing-honest-claims` asks for: read the bold sentence alone, out of
context, and see if it says more than the table shows. This one did -- and the fix wasn't just
softer words, it caught a claim that did not survive a second look at its own direction.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.